# 12 — Lea County, NM: Multi-Source Ensemble Time Series (2020–2022)

Comprehensive field boundary delineation using **all available satellite products and engines** with vote-based ensemble merging.

**Sources:** Sentinel-2, Landsat, HLS, NAIP, SPOT, Google & TESSERA embeddings
**Engines:** delineate-anything, FTW, GeoAI, Prithvi, embedding

**Study area:** Lea County (County 25) from NMOSE WUCB agricultural polygon boundaries. The ensemble runs every compatible source–engine combination per year and merges via majority-vote overlap.

**Estimated runtime:** ~2–4 hours (3 years × up to 15 source–engine combos, GPU recommended).

**Prerequisites:**
```bash
pip install agribound[gee,delineate-anything,ftw,geoai,prithvi]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
import logging
from pathlib import Path

import geopandas as gpd

import agribound
from agribound.evaluate import evaluate

# Enable agribound logging so download/processing progress is visible
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(message)s",
    datefmt="%H:%M:%S",
)

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

NMOSE_SHAPEFILE = "../NMOSE Field Boundaries/WUCB ag polys.shp"
OUTPUT_DIR = Path("../../outputs/lea_county_ensemble")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COUNTY_CODE = 25  # Lea County
YEARS = range(2020, 2023)
VOTE_THRESHOLD = 0.3  # Fraction of source–engine combos that must agree

# Source → compatible engines
SOURCE_ENGINE_MAP = {
    "sentinel2": ["delineate-anything", "ftw", "geoai", "prithvi"],
    "landsat": ["delineate-anything", "ftw", "prithvi"],
    "hls": ["delineate-anything", "ftw", "prithvi"],
    "naip": ["delineate-anything", "geoai"],
    "spot": ["delineate-anything"],
    "google-embedding": ["embedding"],
    "tessera-embedding": ["embedding"],
}

# Year availability constraints
SOURCE_YEAR_RANGE = {
    "sentinel2": (2017, 2025),
    "landsat": (1985, 2025),
    "hls": (2013, 2025),
    "naip": (2003, 2025),
    "spot": (2012, 2023),  # 2012-10-17 to 2023-11-15
    "google-embedding": (2018, 2024),
    "tessera-embedding": (2017, 2024),
}

## Derive Lea County Study Area and Load Reference Boundaries

In [ ]:
full_gdf = gpd.read_file(NMOSE_SHAPEFILE)
ref_gdf = full_gdf[full_gdf["County"] == COUNTY_CODE].copy()
print(f"Lea County: {len(ref_gdf)} reference field polygons loaded")

# Reproject to WGS84 for GeoJSON (bounds must be in lon/lat)
county_4326 = ref_gdf.to_crs(epsg=4326)
bounds = county_4326.total_bounds
bbox_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [bounds[0], bounds[1]],
                        [bounds[2], bounds[1]],
                        [bounds[2], bounds[3]],
                        [bounds[0], bounds[3]],
                        [bounds[0], bounds[1]],
                    ]
                ],
            },
            "properties": {"name": f"Lea County (County {COUNTY_CODE})"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "lea_county_study_area.geojson")
with open(study_area, "w") as f:
    json.dump(bbox_geojson, f)

print(f"Study area: {study_area}")
print(f"Bounds (WGS84): {bounds}")

## Phase 1: Per-Source, Per-Engine Delineation

Run every compatible source–engine combination individually for each year.

In [ ]:
all_results = {}  # {year: {"source/engine": gdf}}

for year in YEARS:
    print(f"\n--- Year {year} ---")
    all_results[year] = {}

    for source, engines in SOURCE_ENGINE_MAP.items():
        yr_min, yr_max = SOURCE_YEAR_RANGE[source]
        if year < yr_min or year > yr_max:
            continue

        for engine in engines:
            tag = f"{source}/{engine}"
            output_path = OUTPUT_DIR / f"fields_{source}_{engine}_{year}.gpkg"

            if output_path.exists():
                print(f"  {tag}: already exists, skipping.")
                all_results[year][tag] = gpd.read_file(output_path)
                continue

            print(f"  {tag}: starting...", flush=True)

            kwargs = dict(
                study_area=study_area,
                source=source,
                year=year,
                engine=engine,
                output_path=str(output_path),
                gee_project=GEE_PROJECT,
                min_area=2500,
                simplify=2.0,
                device="auto",
            )

            # Source-specific composite parameters
            if source in ("sentinel2", "landsat", "hls"):
                kwargs["composite_method"] = "median"
                kwargs["cloud_cover_max"] = 20
            elif source == "spot":
                kwargs["composite_method"] = "median"
                kwargs["cloud_cover_max"] = 15
            elif source == "naip":
                kwargs["min_area"] = 5000
            elif source in ("google-embedding", "tessera-embedding"):
                kwargs["device"] = "cpu"
                kwargs["min_area"] = 5000
                kwargs["engine_params"] = {
                    "use_pca": True,
                    "pca_components": 16,
                    "n_clusters": "auto",
                }

            try:
                gdf = agribound.delineate(**kwargs)
                all_results[year][tag] = gdf
                print(f"  {tag}: {len(gdf)} fields")
            except Exception as exc:
                print(f"  {tag}: FAILED — {exc}")

## Phase 2: Grand Ensemble (Vote Merge)

Merge all source–engine results per year via majority-vote rasterization.

In [ ]:
from agribound.engines.ensemble import EnsembleEngine
from agribound.postprocess import filter_polygons

ensemble_results = {}

for year in YEARS:
    year_results = all_results.get(year, {})
    if len(year_results) < 2:
        print(f"{year}: only {len(year_results)} result(s), skipping ensemble.")
        continue

    output_path = OUTPUT_DIR / f"fields_grand_ensemble_{year}.gpkg"

    if output_path.exists():
        print(f"{year}: already exists, loading.")
        ensemble_results[year] = gpd.read_file(output_path)
        continue

    print(f"{year}: merging {len(year_results)} source–engine results...", end=" ")
    try:
        gdf = EnsembleEngine._merge_vote(year_results, VOTE_THRESHOLD)
        gdf = filter_polygons(gdf, min_area_m2=2500)
        gdf.to_file(output_path, driver="GPKG")
        ensemble_results[year] = gdf
        print(f"{len(gdf)} fields")
    except Exception as exc:
        print(f"FAILED — {exc}")

## Phase 3: Evaluation Against NMOSE Reference (Lea County)

In [ ]:
print(f"{'Year':<6} {'Source/Engine':<40} {'Fields':>6} {'F1':>6} {'IoU':>6} {'P':>6} {'R':>6}")
print(f"{'-' * 6} {'-' * 40} {'-' * 6} {'-' * 6} {'-' * 6} {'-' * 6} {'-' * 6}")

for year in YEARS:
    for tag, gdf in sorted(all_results.get(year, {}).items()):
        try:
            m = evaluate(gdf, ref_gdf)
            print(
                f"{year:<6} {tag:<40} {len(gdf):>6} "
                f"{m['f1']:.3f} {m['iou_mean']:.3f} "
                f"{m['precision']:.3f} {m['recall']:.3f}"
            )
        except Exception:
            pass

    gdf = ensemble_results.get(year)
    if gdf is not None:
        try:
            m = evaluate(gdf, ref_gdf)
            print(
                f"{year:<6} {'** GRAND ENSEMBLE **':<40} {len(gdf):>6} "
                f"{m['f1']:.3f} {m['iou_mean']:.3f} "
                f"{m['precision']:.3f} {m['recall']:.3f}"
            )
        except Exception:
            pass

## Visualization: Grand Ensemble vs NMOSE Reference

In [ ]:
from agribound.visualize import show_comparison

if ensemble_results:
    latest_year = max(ensemble_results.keys())
    m = show_comparison(
        [ensemble_results[latest_year], ref_gdf],
        labels=[f"Grand Ensemble ({latest_year})", "NMOSE Reference"],
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_ensemble_vs_reference.html"),
    )
    m

## Visualization: Source–Engine Comparison

In [ ]:
if ensemble_results:
    latest = all_results.get(latest_year, {})
    if latest:
        comp_gdfs = list(latest.values())
        comp_labels = list(latest.keys())
        comp_gdfs.append(ensemble_results[latest_year])
        comp_labels.append("Grand Ensemble")

        m_engines = show_comparison(
            comp_gdfs,
            labels=comp_labels,
            basemap="Esri.WorldImagery",
            output_html=str(OUTPUT_DIR / "map_source_engine_comparison.html"),
        )
        m_engines

## Visualization: Ensemble Time Series (2020–2022)

In [ ]:
ts_gdfs = [ensemble_results[y] for y in sorted(ensemble_results.keys())]
ts_labels = [str(y) for y in sorted(ensemble_results.keys())]

if len(ts_gdfs) >= 2:
    m_ts = show_comparison(
        ts_gdfs,
        labels=ts_labels,
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_ensemble_timeseries.html"),
    )
    m_ts